# Credit Fraud Mentoring Walkthrough

Thin notebook: **only calls package functions**. Logic lives under `src/credit_fraud/`.

Prerequisite (repo root):

```bash
pip install -e ".[dev]"
```

In [ ]:
from credit_fraud import config
from credit_fraud.data import load_and_split
from credit_fraud.eda import run_eda
from credit_fraud.rank_plot import rank_and_plot
from credit_fraud.features import build_features
from credit_fraud.benchmark import run_benchmark
from credit_fraud.optimize import optimize_models
from credit_fraud.selection import select_best

print("TARGET_COL =", config.TARGET_COL)
print("MAX_ROWS   =", config.MAX_ROWS)
print("N_TRIALS   =", config.N_TRIALS)
print("DATA_PATH  =", config.DATA_PATH)

## 1) Load + stratified split

In [ ]:
x_train, x_test, y_train, y_test, raw = load_and_split()
x_train.shape, y_train.mean()

## 2) EDA (independent + dependent)

In [ ]:
eda = run_eda(x_train, y_train, save=True)
display(eda.target_summary)
print("imbalance neg/pos =", round(eda.imbalance_ratio, 2))
eda.figures["target_distribution"]

## 3) Rank & plot

In [ ]:
ranks = rank_and_plot(x_train, y_train, save=True)
display(ranks.ranking.head(10))
ranks.figures["rank_bar"]

## 4–5) Features + benchmark

In [ ]:
x_train_fe, x_test_fe, builder = build_features(x_train, x_test, y_train)
bench = run_benchmark(x_train_fe, x_test_fe, y_train, y_test)
bench.metrics

## 6–7) Optuna (RF + XGBoost) → selection

Uses `config.N_TRIALS`. For a quicker cell, pass `n_trials=5`.

In [ ]:
opt = optimize_models(x_train_fe, x_test_fe, y_train, y_test)  # or n_trials=5
selection = select_best(
    model_metrics=opt.test_metrics,
    models=opt.best_models,
    feature_names=list(x_train_fe.columns),
    benchmark_metrics=bench.metrics,
    cv_scores=opt.cv_scores,
    best_params=opt.best_params,
    save=True,
)
display(selection.comparison)
print("winner:", selection.best_name)
selection.importances.head(15)

## One-liner alternative

```python
from credit_fraud.pipeline import run_training
result = run_training()  # or run_training(n_trials=5)
result.comparison
```